# Chấm lại bằng Qwen3-Reranker-8B (mô hình thứ hai) — đo thời gian rồi chấm theo khoảng câu hỏi

**Chuẩn bị:** Settings → Accelerator **GPU T4 x2**, Internet **On**.
Add Input: dataset chứa `pairs.parquet` xuất bằng `prf_rerank.py --export-band ...` (đúng khoá cặp với điểm 4B).
Tuỳ chọn: Add Input thêm `scores.parquet` 4B của các lượt cũ để notebook so sánh 8B với 4B trên cùng các cặp.
Chạy tiếp lần trước: Add Input thêm output của lần đó (`scores_8b.parquet` / `scores8b_gpu*.jsonl`).

**Lần đầu: chạy thử** `MAX_QUERIES = 20` để đo tốc độ (cặp/giây) và xem 8B khác 4B bao nhiêu.
**Chấm thật:** `MAX_QUERIES = 0`, chia câu cho các tài khoản bằng `QID_FROM`/`QID_TO` (khoảng id câu hỏi) hoặc
`SHARD_INDEX`/`SHARD_COUNT`.

**Kết quả:** `/kaggle/working/scores_8b.parquet` (gồm cả điểm của các lượt cũ đã gắn vào) → tải output của TỪNG bản về,
đặt vào `work/rerank_8b_band/` với tên khác nhau bắt đầu bằng `scores_8b` (vd. `scores_8b_1.parquet`, `scores_8b_2.parquet`),
rồi dựng bài bằng `prf_rerank.py --scores8b "work/rerank_8b_band/scores_8b*.parquet"`.

In [ ]:
# ===== CẤU HÌNH =====
MODEL_ID = 'Qwen/Qwen3-Reranker-8B'
QUANT = '4bit'         # '4bit' (~5,5 GB/GPU, mỗi GPU một tiến trình), '8bit' (~9 GB) hoặc 'none' (chỉ cho 4B)
MAX_QUERIES = 20       # chạy thử: chỉ N câu đầu (theo id) trong phần việc này; 0 = tất cả
QID_FROM, QID_TO = None, None    # chia theo khoảng id câu hỏi, vd. 1, 600 (tính cả hai đầu); None = không lọc
SHARD_INDEX, SHARD_COUNT = 0, 1  # hoặc chia theo id % SHARD_COUNT
MAX_LENGTH = 768       # giữ bằng lượt chấm 4B để hai điểm so được với nhau
BATCH_SIZE = 8         # tự giảm khi hết VRAM
TIME_LIMIT_HOURS = 11.0
DUMMY = False          # True: chạy thử đường ống, không cần GPU/model

In [ ]:
import glob, json, os, subprocess, sys, threading, time
START = time.time()
WORK = '/kaggle/working'
INPUT = '/kaggle/input'

found = sorted(glob.glob(f'{INPUT}/**/pairs.parquet', recursive=True))
if not found:
    raise FileNotFoundError('Chưa gắn dataset chứa pairs.parquet (Add Input).')
PAIRS = found[0]
RESUME = sorted(glob.glob(f'{INPUT}/**/scores_8b*.parquet', recursive=True)
                + glob.glob(f'{INPUT}/**/scores8b_gpu*.jsonl', recursive=True))
REF4B = [p for p in sorted(glob.glob(f'{INPUT}/**/scores*.parquet', recursive=True)
                           + glob.glob(f'{INPUT}/**/scores_gpu*.jsonl', recursive=True)) if p not in RESUME]
print('PAIRS =', PAIRS)
print('Kết quả 8B cũ để bỏ qua:', RESUME or 'không có')
print('Điểm 4B để so sánh:', REF4B or 'không có')

if DUMMY:
    N_GPU = 2
else:
    import torch, transformers
    N_GPU = torch.cuda.device_count()
    print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', N_GPU,
          [torch.cuda.get_device_name(i) for i in range(N_GPU)])
    if N_GPU == 0:
        raise RuntimeError('Không có GPU: Settings -> Accelerator -> GPU T4 x2.')
    major, minor = (int(x) for x in transformers.__version__.split('.')[:2])
    if (major, minor) < (4, 51):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51'], check=True)
    if QUANT != 'none':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.43'], check=True)
        print('Đã cài bitsandbytes')

In [ ]:
WORKER_SRC = '"""Chấm lại các cặp (câu hỏi, đoạn) bằng Qwen3-Reranker trên MỘT GPU (bước A2, chạy trên Kaggle).\n\nĐọc một file parquet các cặp (qid, query, doc_id, run, chunk_order, text), ghi từng dòng JSON\n{"key": ..., "score": ...} vào --out và flush sau mỗi batch, nên dừng giữa chừng không mất gì; chạy\nlại với cùng --out thì bỏ qua các cặp đã chấm. Prompt và cách tính điểm giống hệt reranker Qwen3 của\npipeline (src/vibiomir/retrieval/rerank.py): xác suất "yes" so với "no" ở token cuối.\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport re\nimport sys\nimport time\n\nimport pyarrow.parquet as pq\n\nINSTRUCTION = (\'Given a Vietnamese medical question, judge whether the document passage \'\n               \'contains information that answers it.\')\nPREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n          \'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n\'\n          \'<|im_start|>user\\n\')\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef pair_key(qid, doc_id, run, order):\n    return f\'{int(qid)}:{int(doc_id)}:{run}:{int(order)}\'\n\n\ndef done_keys(path):\n    keys = set()\n    if os.path.exists(path):\n        with open(path, encoding=\'utf-8\') as stream:\n            for line in stream:\n                try:\n                    keys.add(json.loads(line)[\'key\'])\n                except (ValueError, KeyError):\n                    pass  # A line cut by a crash; that pair is simply scored again.\n    return keys\n\n\nclass DummyScorer:\n    """Word overlap, for testing the plumbing without a GPU or a model download."""\n\n    def score(self, queries, passages):\n        out = []\n        for query, passage in zip(queries, passages):\n            q = set(re.findall(r\'\\w+\', query.lower()))\n            p = set(re.findall(r\'\\w+\', passage.lower()))\n            out.append(len(q & p) / max(1, len(q)))\n        return out\n\n\nclass QwenScorer:\n    def __init__(self, model_id, max_length, quant=\'none\'):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        self.torch = torch\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side=\'left\')\n        if quant == \'none\':\n            self.model = AutoModelForCausalLM.from_pretrained(\n                model_id, torch_dtype=torch.float16, attn_implementation=\'sdpa\').to(\'cuda\').eval()\n        else:\n            # 8B in fp16 (~16 GB) does not fit one T4: load 4-bit (~5.5 GB) or 8-bit (~9 GB) with bitsandbytes.\n            from transformers import BitsAndBytesConfig\n            config = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)\n                      if quant == \'4bit\' else BitsAndBytesConfig(load_in_8bit=True))\n            self.model = AutoModelForCausalLM.from_pretrained(\n                model_id, quantization_config=config, torch_dtype=torch.float16, attn_implementation=\'sdpa\',\n                device_map={\'\': 0}).eval()\n        self.yes_id = self.tokenizer.convert_tokens_to_ids(\'yes\')\n        self.no_id = self.tokenizer.convert_tokens_to_ids(\'no\')\n        self.prefix_ids = self.tokenizer.encode(PREFIX, add_special_tokens=False)\n        self.suffix_ids = self.tokenizer.encode(SUFFIX, add_special_tokens=False)\n        self.budget = max_length - len(self.prefix_ids) - len(self.suffix_ids)\n\n    def score(self, queries, passages):\n        prompts = [f\'<Instruct>: {INSTRUCTION}\\n<Query>: {q}\\n<Document>: {p}\'\n                   for q, p in zip(queries, passages)]\n        encoded = self.tokenizer(prompts, padding=False, truncation=True, add_special_tokens=False,\n                                 return_attention_mask=False, max_length=self.budget)\n        encoded[\'input_ids\'] = [self.prefix_ids + ids + self.suffix_ids for ids in encoded[\'input_ids\']]\n        inputs = self.tokenizer.pad(encoded, padding=True, return_tensors=\'pt\').to(\'cuda\')\n        with self.torch.inference_mode():\n            last = self.model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :]\n            pair = self.torch.stack([last[:, self.no_id], last[:, self.yes_id]], dim=1).float()\n            return self.torch.softmax(pair, dim=1)[:, 1].cpu().tolist()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument(\'--pairs\', required=True)\n    parser.add_argument(\'--out\', required=True)\n    parser.add_argument(\'--model\', default=\'Qwen/Qwen3-Reranker-4B\')\n    parser.add_argument(\'--max-length\', type=int, default=768)\n    parser.add_argument(\'--batch-size\', type=int, default=16)\n    parser.add_argument(\'--deadline\', type=float, default=0, help=\'Unix time to stop at; 0 = none.\')\n    parser.add_argument(\'--tag\', default=\'\')\n    parser.add_argument(\'--dummy\', action=\'store_true\')\n    parser.add_argument(\'--quant\', choices=[\'none\', \'8bit\', \'4bit\'], default=\'none\',\n                        help=\'Nạp mô hình lượng tử hoá (cần bitsandbytes); dùng cho 8B trên T4.\')\n    args = parser.parse_args()\n\n    table = pq.read_table(args.pairs, columns=[\'qid\', \'query\', \'doc_id\', \'run\', \'chunk_order\', \'text\'])\n    rows = list(zip(*(table[name].to_pylist() for name in table.column_names)))\n    done = done_keys(args.out)\n    todo = [row for row in rows if pair_key(row[0], row[2], row[3], row[4]) not in done]\n    # Longest first: an out-of-memory batch shows up at the start and the batch size settles there.\n    todo.sort(key=lambda row: -(len(row[1]) + len(row[5])))\n    print(f\'{args.tag} cặp: {len(rows)}, đã chấm: {len(rows) - len(todo)}, còn: {len(todo)}\', flush=True)\n    if not todo:\n        return\n    scorer = DummyScorer() if args.dummy else QwenScorer(args.model, args.max_length, args.quant)\n    size, position, written, bad = args.batch_size, 0, 0, 0\n    started = last_report = time.time()\n    with open(args.out, \'a\', encoding=\'utf-8\') as out:\n        while position < len(todo):\n            if args.deadline and time.time() > args.deadline:\n                print(f\'{args.tag} hết giờ, dừng ở {position}/{len(todo)}; chạy lại để làm tiếp.\', flush=True)\n                break\n            batch = todo[position:position + size]\n            try:\n                scores = scorer.score([r[1] for r in batch], [r[5] for r in batch])\n            except RuntimeError as error:\n                if \'out of memory\' not in str(error).lower() or size == 1:\n                    raise\n                size = max(1, size // 2)\n                scorer.torch.cuda.empty_cache()\n                print(f\'{args.tag} hết VRAM, giảm batch còn {size}\', flush=True)\n                continue\n            for row, score in zip(batch, scores):\n                if score is None or not math.isfinite(score):\n                    bad += 1\n                    score = None\n                out.write(json.dumps({\'key\': pair_key(row[0], row[2], row[3], row[4]), \'score\': score}) + \'\\n\')\n            out.flush()\n            position += len(batch)\n            written += len(batch)\n            if written >= 200 and bad > written // 2:\n                raise RuntimeError(f\'{bad}/{written} điểm không hợp lệ (NaN/inf) với fp16. \'\n                                   \'Mô hình này không chạy được fp16 trên GPU hiện tại; đổi MODEL_ID.\')\n            now = time.time()\n            if now - last_report > 60 or position >= len(todo):\n                rate = written / max(1e-9, now - started)\n                eta = (len(todo) - position) / max(rate, 1e-9) / 60\n                print(f\'{args.tag} {position}/{len(todo)} cặp, {rate:.1f} cặp/s, còn ~{eta:.0f} phút, \'\n                      f\'batch {size}, điểm lỗi {bad}\', flush=True)\n                last_report = now\n    print(f\'{args.tag} xong {written} cặp trong {(time.time() - started) / 60:.1f} phút, điểm lỗi {bad}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    main()\n'
with open(f'{WORK}/rerank_worker.py', 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('Đã ghi rerank_worker.py')

In [ ]:
import numpy as np
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq

ALL = pq.read_table(PAIRS)
qids = ALL['qid'].to_numpy()
mask = (qids % SHARD_COUNT == SHARD_INDEX)
if QID_FROM is not None:
    mask &= qids >= QID_FROM
if QID_TO is not None:
    mask &= qids <= QID_TO
table = ALL.filter(pa.array(mask))
FULL_ROWS = table.num_rows            # cặp của phần việc này khi chấm thật (để ước tính thời gian)
if MAX_QUERIES:
    keep = set(sorted(set(table['qid'].to_pylist()))[:MAX_QUERIES])
    table = table.filter(pc.is_in(table['qid'], value_set=pa.array(sorted(keep), type=table['qid'].type)))

def key_of(q, d, r, o):
    return f'{int(q)}:{int(d)}:{r}:{int(o)}'

def read_scores(paths):
    out = {}
    for path in paths:
        if path.endswith('.parquet'):
            t = pq.read_table(path, columns=['key', 'score'])
            items = zip(t['key'].to_pylist(), t['score'].to_pylist())
        else:
            items = []
            with open(path, encoding='utf-8') as f:
                for line in f:
                    try:
                        row = json.loads(line)
                    except ValueError:
                        continue
                    items.append((row.get('key'), row.get('score')))
        for k, s in items:
            if k is not None and s is not None:
                out[k] = s
    return out

done = read_scores(RESUME)
keys = [key_of(*r) for r in zip(*(table[c].to_pylist() for c in ('qid', 'doc_id', 'run', 'chunk_order')))]
todo = table.filter(pa.array([k not in done for k in keys]))
print(f'Cả file: {ALL.num_rows:,} cặp | phần việc này (khi chấm thật): {FULL_ROWS:,} | lần chạy này: {table.num_rows:,} '
      f'({len(set(table["qid"].to_pylist()))} câu) | còn phải chấm: {todo.num_rows:,}')

lengths = pc.add(pc.utf8_length(todo['text']), pc.utf8_length(todo['query'])).to_pylist()
order = sorted(range(todo.num_rows), key=lambda i: -lengths[i])
PARTS = []
for g in range(N_GPU):
    part = todo.take(pa.array(order[g::N_GPU], type=pa.int64()))
    path = f'{WORK}/part{g}.parquet'
    pq.write_table(part, path)
    PARTS.append(path)
    print(f'GPU{g}: {part.num_rows:,} cặp')

In [ ]:
if not DUMMY and todo.num_rows:
    # Thanh tiến trình khi tải model làm tràn kênh output, papermill báo "Timeout waiting for IOPub output" và dừng
    # cả bản chạy (Save & Run All): tắt đi.
    os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
    from huggingface_hub import snapshot_download
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
    t0 = time.time()
    snapshot_download(MODEL_ID)
    print(f'Đã tải {MODEL_ID} trong {(time.time() - t0) / 60:.1f} phút')

In [ ]:
deadline = START + TIME_LIMIT_HOURS * 3600
procs = []

def pump(proc):
    for line in proc.stdout:
        print(line, end='', flush=True)

SCORE_START = time.time()
for g, part in enumerate(PARTS):
    cmd = [sys.executable, '-u', f'{WORK}/rerank_worker.py', '--pairs', part, '--out', f'{WORK}/scores8b_gpu{g}.jsonl',
           '--model', MODEL_ID, '--max-length', str(MAX_LENGTH), '--batch-size', str(BATCH_SIZE),
           '--deadline', str(deadline), '--tag', f'[GPU{g}]', '--quant', QUANT] + (['--dummy'] if DUMMY else [])
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), TOKENIZERS_PARALLELISM='false')
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    threading.Thread(target=pump, args=(proc,), daemon=True).start()
    procs.append(proc)
codes = [proc.wait() for proc in procs]
SCORE_SECONDS = time.time() - SCORE_START
time.sleep(2)
print('Mã thoát:', codes)

In [ ]:
scores = dict(done)
scores.update(read_scores(sorted(glob.glob(f'{WORK}/scores8b_gpu*.jsonl'))))
pq.write_table(pa.table({'key': list(scores), 'score': pa.array(list(scores.values()), type=pa.float32())}),
               f'{WORK}/scores_8b.parquet')
need = set(keys)
have = sum(1 for k in need if k in scores)
print(f'scores_8b.parquet: {len(scores):,} cặp | lần chạy này: {have:,}/{len(need):,} ({100 * have / max(1, len(need)):.1f}%)')

# Tốc độ (gồm cả thời gian nạp mô hình) và ước tính cho phần việc khi chấm thật
new_pairs = todo.num_rows
if new_pairs and SCORE_SECONDS > 0:
    rate = new_pairs / SCORE_SECONDS
    print(f'Tốc độ: {rate:.2f} cặp/giây cho cả {N_GPU} GPU (kể cả nạp mô hình) | '
          f'chấm thật {FULL_ROWS:,} cặp mất ~{FULL_ROWS / rate / 3600:.1f} giờ cho phiên này')

# So sánh với 4B trên cùng các cặp
ref = read_scores(REF4B)
both = [(ref[k], scores[k]) for k in need if k in ref and k in scores]
if both:
    a = np.array([x for x, _y in both]); b = np.array([y for _x, y in both])
    def spearman(x, y):
        rx = np.argsort(np.argsort(x)); ry = np.argsort(np.argsort(y))
        return float(np.corrcoef(rx, ry)[0, 1])
    print(f'So với 4B trên {len(both):,} cặp: Spearman {spearman(a, b):.3f}, Pearson {np.corrcoef(a, b)[0, 1]:.3f}, '
          f'|8B-4B| trung bình {np.mean(np.abs(a - b)):.3f}, lệch > 0.3: {100 * np.mean(np.abs(a - b) > 0.3):.1f}%')
    by_q = {}
    for k in need:
        if k in ref and k in scores:
            by_q.setdefault(int(k.split(':')[0]), []).append((ref[k], scores[k]))
    per_q = [spearman(np.array([x for x, _ in v]), np.array([y for _, y in v])) for v in by_q.values() if len(v) >= 10]
    if per_q:
        print(f'Spearman trong từng câu: trung vị {np.median(per_q):.3f}, p10 {np.percentile(per_q, 10):.3f}, p90 {np.percentile(per_q, 90):.3f}')
    print('Gợi ý: Spearman > 0,95 thì hai mô hình gần như xếp giống nhau (lấy trung bình ít lợi); 0,7–0,9 là mức đáng thử.')
else:
    print('Không có điểm 4B trùng khoá để so sánh (Add Input scores.parquet 4B nếu muốn).')
print(f'Tổng thời gian: {(time.time() - START) / 60:.1f} phút')